# Tutorial for ANONLINK

Based on ANONLINK

First install anonlink-client, clkhash, recordlinkage and a few data science tools (pandas and numpy):

In [1]:
%pip install -U anonlink-client clkhash anonlink recordlinkage numpy pandas

  Using cached recordlinkage-0.16-py3-none-any.whl.metadata (8.1 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached cython-3.2.4-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (7.5 kB)
  Using cached metaphone-0.6-py3-none-any.whl
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached cffi-2.0.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.6 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached charset_normalizer-3.4.6-cp310-cp310-manylinux2014_x86_64.manylinux_2_17

## Imports

In [2]:
import io
import numpy as np
import pandas as pd
import itertools

# import modules necessary for schema definition

from clkhash.clk import generate_clk_from_csv

from blocklib import generate_candidate_blocks

from clkhash.field_formats import FieldHashingProperties, Ignore, BitsPerFeatureStrategy, StringSpec
from clkhash.comparators import NgramComparison

# import clkhash
# from clkhash.field_formats import *
from clkhash.schema import Schema
# from clkhash.comparators import NgramComparison

# from anonlinkclient.utils import generate_clk_from_csv, generate_candidate_blocks_from_csv, combine_clks_blocks

import recordlinkage

## Load ABT-BUY Dataset

In [8]:
## Load Dataset

abt = pd.read_csv("../data/D2/abtclean.csv", sep="|")
buy = pd.read_csv("../data/D2/buyclean.csv", sep="|")

abt_csv = io.StringIO()
abt.to_csv(abt_csv, index=False)

buy_csv = io.StringIO()
buy.to_csv(buy_csv, index=False)

abt.head(10)


,id,name,description,price
0,0,Sony Turntable - PSLX350H,Sony Turntable - PSLX350H/ Belt Drive System/ ...,NaN
1,1,Bose Acoustimass 5 Series III Speaker System -...,Bose Acoustimass 5 Series III Speaker System -...,399.0
2,2,Sony Switcher - SBV40S,Sony Switcher - SBV40S/ Eliminates Disconnecti...,49.0
3,3,Sony 5 Disc CD Player - CDPCE375,Sony 5 Disc CD Player- CDPCE375/ 5 Disc Change...,NaN
4,4,Bose 27028 161 Bookshelf Pair Speakers In Whit...,Bose 161 Bookshelf Speakers In White - 161WH/ ...,158.0
5,5,Denon Stereo Tuner - TU1500RD,Denon Stereo Tuner - TU1500RD/ RDS Radio Data ...,375.0
6,6,KitchenAid Pasta Roller And Cutter - KPRA,KitchenAid Pasta Roller And Cutter - KPRA/ One...,NaN
7,7,Panasonic Yeast Pro Automatic Breadmaker - SDY...,Panasonic Yeast Pro Automatic Breadmaker - SDY...,NaN
8,8,Sony Vertical-In-The-Ear Stereo Headphones - M...,Sony Vertical-In-The-Ear Stereo Headphones - M...,NaN
9,9,Panasonic 2-Line Integrated Telephone - KXTSC14W,Panasonic 2-Line Integrated Telephone - KXTSC1...,NaN


## Hashing Schema Definition

In [11]:
fields = [
    Ignore('id'),
    StringSpec('name', FieldHashingProperties(comparator=NgramComparison(2), strategy=BitsPerFeatureStrategy(300))),
    Ignore('description'),
    Ignore('price'), 
]

schema = Schema(fields, 1024)

## Hash the Data

In [14]:
secret = 'secret'

abt_csv.seek(0)

clks_abt = generate_clk_from_csv(abt_csv, secret, schema)

clks_abt[:10]

generating CLKs: 100%|█████████▉| 1.08k/1.08k [00:00<00:00, 55.7kclk/s, mean=235, std=14.2]


[bitarray('00001000001000101010001010011000100110000000000100111000000100000000010100001000010000100010100000001001001110001000000010000001010001001010001101101111100000100000010011000100000001011001000000000000000011101010000010010000010000111111000000000000100111000000100100100001010000100100000100100011000000110000000001001001000100000000100001001100011100010100010000000000000000001000001100011101000000010111010100110000100000001000100101000000011000000001000101101010000000100001100000001001100000000000001000001000110110001010000010001010000100000110000000010000000000000000001000100000010100001000000000011101001100011000110100010001000000100001100000010000100010000101000001000000000000011010011000001001010100001100010010100000011000010000000010000010010000010000010000000000100010001010001000000000000010000000000010001110100101001000000100010000000100110000000000101001000000000010000000101000000001000010001010001101100010100000010001000000000011000000100000100001100000001001000000010

In [15]:
secret = 'secret'

buy_csv.seek(0)

clks_buy = generate_clk_from_csv(buy_csv, secret, schema)

clks_buy[:10]

generating CLKs: 100%|█████████▉| 1.06k/1.06k [00:00<00:00, 12.3kclk/s, mean=234, std=17.3]


[bitarray('00101000000010100000000010000000001010000000001000000011001001000000000100000001001000001000000000000000000100000010100000000010100000000000100000011000100000000000000010000001000000000000000000000000100101110001000000010100000110001000000100000000010000000000100000100000000000000100010000000110000010000000000000100000010010000100001000000100000000100000000001001000000110001001010000100100011001000110000000100001000000010000011000010000000010000100000000011100000000000010000001001001000000000000000000000010010000000011000001010101000000100000000000000010000000000010000001001010100000000000100000001000100000001000000000000110000001000001011000000010100100000000101000000010001000011000000001001001000001001000001000000001000100000100001010001010000000010010000100000001100010000000010110100000010000001100000010000000001010000000000100000100100000010000010010010100100000001010101000000010000000000000000010000100000110100000010000010101010100100000000000000010000000100010000000000

## Matching

In [25]:
from anonlinkclient.utils import deserialize_filters
from bitarray import bitarray
import base64

import anonlink

def mapping_from_clks(clks_a, clks_b, threshold):
    results_candidate_pairs = anonlink.candidate_generation.find_candidate_pairs(
            [clks_a, clks_b],
            anonlink.similarities.dice_coefficient,
            threshold
    )
    solution = anonlink.solving.greedy_solve(results_candidate_pairs)
    print('Found {} matches'.format(len(solution)))
    # each entry in `solution` looks like this: '((0, 4039), (1, 2689))'.
    # The format is ((dataset_id, row_id), (dataset_id, row_id))
    # As we only have two parties in this example, we can remove the dataset_ids.
    # Also, turning the solution into a set will make it easier to assess the
    # quality of the matching.
    return set((a, b) for ((_, a), (_, b)) in solution)

found_matches = mapping_from_clks(clks_abt, clks_buy, 0.6)

Found 753 matches


In [26]:
list(found_matches)[:10]

[(683, 718),
 (251, 405),
 (644, 708),
 (1014, 427),
 (6, 210),
 (352, 289),
 (899, 934),
 (44, 191),
 (670, 882),
 (939, 927)]

## Evaluation

In [27]:
ground_truth = pd.read_csv("../data/D2/gtclean.csv", sep='|')

true_positives = 0
total_matching_pairs = len(found_matches)

for _, (id1, id2) in ground_truth.iterrows():

    if (id1, id2) in found_matches:
        true_positives += 1

num_of_true_duplicates = ground_truth.shape[0]
false_negatives = num_of_true_duplicates - true_positives
false_positives = total_matching_pairs - true_positives
cardinality = abt.shape[0] * buy.shape[0]
true_negatives = cardinality - false_negatives - num_of_true_duplicates
precision = true_positives / total_matching_pairs
recall = true_positives / num_of_true_duplicates
if precision == 0.0 or recall == 0.0:
    f1 = 0.0
else:
    f1 = 2*((precision*recall)/(precision+recall))

print(f"Precision: {precision}\nRecall: {recall}\nF1: {f1}")

Precision: 0.3612217795484728
Recall: 0.2556390977443609
F1: 0.2993946064942212
